In [1]:
from convertiq_py.ml_logic.data import load_data_kaggle_raw, clean_data
from convertiq_py.ml_logic.preprocessor import preprocess_features, feature_engineering
from convertiq_py.ml_logic.model import train_test_split, initialize_model, train_model, base_spw
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb

In [2]:
df = load_data_kaggle_raw(10000000)

In [3]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.200001,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.099976,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [4]:
X = clean_data(df)

/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/data.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["has_valid_price"] = (df["price"] > 0).astype("int8")


✅ Data cleaned


In [5]:


# Observation and prediction set split
observation_end = pd.Timestamp("2019-10-06")
prediction_end  = pd.Timestamp("2019-10-08")
X_obs = X[X["event_time"] < observation_end].copy()
X_pred = X[X["event_time"] >= observation_end].copy()

In [6]:
X_obs_2 = X_obs.set_index('user_id')

In [7]:
X_obs_2

,event_time,event_type,product_id,category_id,category_code,brand,price,user_session,has_brand,has_valid_price
user_id,,,,,,,,,,
550050854,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,7c90fc70-0e80-4590-96f3-13c02c18c713,1,1
535871217,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,1,1
537918940,2019-10-01 00:00:11,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.010010,406c46ed-90a4-4787-a43b-59a410c1a5fb,1,1
530282093,2019-10-01 00:00:11,view,1005011,2053013555631882655,electronics.smartphone,samsung,900.640015,50a293fb-5940-41b2-baf3-17af0e812101,1,1
537192226,2019-10-01 00:00:18,view,1801995,2053013554415534427,electronics.video.tv,haier,193.029999,e3151795-c355-4efa-acf6-e1fe1bebeee5,1,1
...,...,...,...,...,...,...,...,...,...,...
552859242,2019-10-05 23:59:52,view,1004750,2053013555631882655,electronics.smartphone,samsung,197.789993,ac93027f-3447-4feb-afb3-83a79b74bc8f,1,1
512536825,2019-10-05 23:59:54,view,1004873,2053013555631882655,electronics.smartphone,samsung,380.309998,ff044704-7e4e-40ea-bebc-ec6690b6d5ce,1,1
550697012,2019-10-05 23:59:55,view,1005115,2053013555631882655,electronics.smartphone,apple,975.559998,41709209-2300-49d2-ae3e-c76be02a5731,1,1


In [13]:
X_feat_eng = feature_engineering(X_obs)


int64


In [14]:
X_feat_eng.head()

,total_events,total_views,total_carts,total_purchases,n_sessions,n_days_active,has_ever_carted,has_ever_purchased,view_to_cart_ratio,cart_to_purchase_ratio,avg_session_duration,median_session_duration,max_session_duration,avg_events_per_session,max_events_per_session
user_id,,,,,,,,,,,,,,,
241587569,1,1,0,0,1,1,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1
241784978,1,1,0,0,1,1,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1
244951053,5,5,0,0,2,2,0,0,0.0,0.0,91.0,91.0,129.0,2.5,3
258811396,3,3,0,0,1,1,0,0,0.0,0.0,245.0,245.0,245.0,3.0,3
263428680,1,1,0,0,1,1,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1


In [15]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test=train_test_split(X_feat_eng, y_purchasers, random_state=42, stratify=y_purchasers)

In [ ]:
import pickle
import joblib
# path = os.path.join('convertiq', 'raw_data')
model = joblib.load('model_lgbm_baseline.pkl')


In [ ]:
df_prediction=pd.read_csv('dataset_2user.csv')


In [49]:
df_clean = clean_data(df_prediction)

✅ Data cleaned


In [50]:
X_processed = feature_engineering(df_clean)

int64


In [51]:
X_processed.head()

,total_events,total_views,total_carts,total_purchases,n_sessions,n_days_active,has_ever_carted,has_ever_purchased,view_to_cart_ratio,cart_to_purchase_ratio,avg_session_duration,median_session_duration,max_session_duration,avg_events_per_session,max_events_per_session
user_id,,,,,,,,,,,,,,,
550050854,7,7,0,0,1,1,0,0,0.000000,0.00,196.000000,196.0,196.0,7.000000,7
551377651,64,51,4,9,9,3,1,1,0.078431,2.25,1562.666667,332.0,9700.0,7.111111,22


In [52]:
y_pred = model.predict(X_processed)

In [ ]:
# Séparer user_id avant de passer au modèle
X_processed = X_processed.reset_index()

user_ids = X_processed['user_id']


In [63]:
user_ids.head()

0    550050854
1    551377651
Name: user_id, dtype: int64

In [73]:
X= X_processed.drop(columns=['user_id'])

y_pred = model.predict(X)
y_proba = model.predict_proba(X)[1]

In [74]:
y_proba

array([0.0208443, 0.9791557])

In [75]:
result = pd.DataFrame({
    'user_id': user_ids,
    'prediction': y_pred,
    'probabilité' : y_proba
})

In [66]:
result

,user_id,prediction
0,550050854,0
1,551377651,1


In [77]:
user_ids[0]

np.int64(550050854)

In [79]:
{
    'user_id': int(user_ids[0]),
    'prediction': int(y_pred[0]),
    'probabilité' : float(y_proba[0])
}

{'user_id': 550050854, 'prediction': 0, 'probabilité': 0.020844295347221098}

In [16]:


# X_pred sert juste à aller récupérer les acheteurs (y) pendant cette période


# Create y with purchasers from prediction period
purchasers = set(X_pred.loc[X_pred["event_type"] == "purchase", "user_id"])
y_purchasers = pd.DataFrame({"user_id": X_obs["user_id"].unique()})
y_purchasers["label"] = y_purchasers["user_id"].isin(purchasers).astype(int)

In [23]:
y_purchasers["label"].nunique()

2

In [20]:
y_purchasers.sum()

user_id    200327341356749
label                 9386
dtype: int64

In [22]:
len(y_purchasers) - y_purchasers.sum()

user_id   -200327340982248
label               365115
dtype: int64

In [25]:
y_purchasers.head()

,user_id,label
0,550050854,0
1,535871217,0
2,537918940,0
3,530282093,0
4,537192226,0


In [29]:
y=y_purchasers.set_index('user_id')

In [30]:
y.head()

,label
user_id,
550050854,0
535871217,0
537918940,0
530282093,0
537192226,0


In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X_feat_eng, y_purchasers, test_size=0.2, random_state=42, stratify=y_purchasers['label'])


In [58]:
y_test

,user_id,label
119335,555933143,0
269247,536197311,0
224233,512562822,0
117192,551960257,0
61493,529380432,0
...,...,...
63725,545204195,0
306991,514116682,0
146123,516010885,0
130478,555981960,0


In [59]:
y_1 = y_test[y_test['label']==1].iloc[-1,:]

In [62]:
y_0 = y_test[y_test['label']==0].iloc[-1,:]

In [63]:
list_user_prediction = [int(y_0['user_id']),int(y_1['user_id'])]

In [64]:
list_user_prediction

[522054597, 557170240]

In [65]:
df_prediction = df[df['user_id'].isin(list_user_prediction)]

In [66]:
df_prediction.shape

(11, 10)

In [67]:
df_prediction.sort_values('user_id')

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,has_brand
4093315,2019-10-04 09:46:55,view,1005115,2053013555631882655,electronics.smartphone,apple,975.559998,522054597,a12e8a58-e321-4877-bd11-676796d7d35e,1
4094156,2019-10-04 09:47:30,view,1005105,2053013555631882655,electronics.smartphone,apple,1415.479980,522054597,a12e8a58-e321-4877-bd11-676796d7d35e,1
6277765,2019-10-05 21:04:55,view,1004708,2053013555631882655,electronics.smartphone,huawei,153.860001,557170240,8bebb6bb-6235-4b53-be7a-2956d1b0a0c0,1
6278281,2019-10-05 21:06:25,cart,1004708,2053013555631882655,electronics.smartphone,huawei,153.860001,557170240,8bebb6bb-6235-4b53-be7a-2956d1b0a0c0,1
7857098,2019-10-07 06:09:19,view,1004709,2053013555631882655,electronics.smartphone,huawei,152.009995,557170240,12304e03-32a1-4f17-9cfb-7a914fbaa238,1
7858301,2019-10-07 06:10:31,view,1004748,2053013555631882655,electronics.smartphone,huawei,130.479996,557170240,12304e03-32a1-4f17-9cfb-7a914fbaa238,1
7865396,2019-10-07 06:17:40,view,1004748,2053013555631882655,electronics.smartphone,huawei,130.479996,557170240,12304e03-32a1-4f17-9cfb-7a914fbaa238,1
8004265,2019-10-07 08:26:19,view,1004720,2053013555631882655,electronics.smartphone,huawei,127.290001,557170240,c7b41aef-8335-43df-9f34-a022897894ca,1
8007969,2019-10-07 08:29:25,purchase,1004720,2053013555631882655,electronics.smartphone,huawei,127.290001,557170240,c7b41aef-8335-43df-9f34-a022897894ca,1
8008998,2019-10-07 08:30:20,view,1004720,2053013555631882655,electronics.smartphone,huawei,127.290001,557170240,c7b41aef-8335-43df-9f34-a022897894ca,1


In [68]:
df_prediction['user_id'].nunique()

2

In [69]:
df_prediction.to_csv('dataset_2user.csv')

In [20]:
df_prediction['event_type_str'] = df_prediction['event_type'].astype(str)
df_prediction.sort_values(by=['user_id', 'user_session', 'product_id', 'event_time'])


/tmp/ipykernel_20036/4263743920.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prediction['event_type_str'] = df_prediction['event_type'].astype(str)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,has_brand,event_type_str
77,2019-10-01 00:01:05,view,1306083,2053013558920217191,computers.notebook,hp,1512.780029,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,1,view
234,2019-10-01 00:03:17,view,1306359,2053013558920217191,computers.notebook,acer,643.489990,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,1,view
18,2019-10-01 00:00:19,view,1306631,2053013558920217191,computers.notebook,hp,580.890015,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,1,view
121,2019-10-01 00:01:42,view,1306631,2053013558920217191,computers.notebook,hp,580.890015,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,1,view
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,1,view
...,...,...,...,...,...,...,...,...,...,...,...
8288822,2019-10-07 13:01:02,view,1200947,2172371436436455782,electronics.tablet,samsung,112.949997,551377651,ce0023cb-5815-4a73-84b4-e55058a4a6c8,1,view
8302494,2019-10-07 13:14:53,view,1201447,2172371436436455782,electronics.tablet,apple,617.520020,551377651,ce0023cb-5815-4a73-84b4-e55058a4a6c8,1,view
163877,2019-10-01 05:33:48,view,1004356,2053013555631882655,electronics.smartphone,apple,1477.260010,551377651,ef960a3a-928f-4f73-851a-fbd20ce0042d,1,view
169814,2019-10-01 05:39:20,view,1307401,2053013558920217191,computers.notebook,asus,306.049988,551377651,ef960a3a-928f-4f73-851a-fbd20ce0042d,1,view


In [21]:
df_seq_prediction = df_prediction.groupby(['user_id', 'user_session', 'category_id']).agg(
    sequence =('event_type_str', lambda x : " ".join(x.iloc[:len(x)]) if len(x)>1 else 'one_view')).reset_index()

In [23]:
df_seq_prediction.head()

,user_id,user_session,category_id,sequence
0,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,2053013558920217191,view view view view view view view
1,551377651,1cd57211-5c6d-4712-ba23-cc5548acb0f4,2053013558920217191,view view view purchase view
2,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,2053013554658804075,view view purchase view
3,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,2053013555631882655,view view view purchase view
4,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,2091727629378912491,view view


In [25]:
X_obs_1 = X_obs[X_obs['user_id']==int(y_1['user_id'])]

In [36]:
X_obs_1.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,has_brand,has_valid_price,hour,dayofweek,is_weekend
12,2019-10-01 00:00:41,view,1003141,2053013555631882655,electronics.smartphone,apple,382.970001,551377651,ca11a570-47da-4630-898b-9a03127703da,1,1,0,1,0
48,2019-10-01 00:01:45,view,1003305,2053013555631882655,electronics.smartphone,apple,612.429993,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,1,1,0,1,0
59,2019-10-01 00:02:08,view,1002528,2053013555631882655,electronics.smartphone,apple,643.229980,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,1,1,0,1,0
79,2019-10-01 00:02:47,view,1002532,2053013555631882655,electronics.smartphone,apple,642.690002,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,1,1,0,1,0
114,2019-10-01 00:04:37,purchase,1002532,2053013555631882655,electronics.smartphone,apple,642.690002,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68,1,1,0,1,0


In [28]:
X_obs_0 = X_obs[X_obs['user_id']==int(y_0['user_id'])]

In [29]:
X_obs_0.shape

(7, 11)

In [30]:
X_prediction_0 = feature_engineering(X_obs_0)

int64


/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/preprocessor.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_obs["hour"] = X_obs["event_time"].dt.hour
/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/preprocessor.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_obs["dayofweek"] = X_obs["event_time"].dt.dayofweek  # 0=Lundi
/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/preprocessor.py:47: SettingWithCopyWarning: 
A value is trying to be set on a c

In [31]:
X_prediction_0.head()

,total_events,total_views,total_carts,total_purchases,n_sessions,n_days_active,has_ever_carted,has_ever_purchased,view_to_cart_ratio,cart_to_purchase_ratio,avg_session_duration,median_session_duration,max_session_duration,avg_events_per_session,max_events_per_session
user_id,,,,,,,,,,,,,,,
550050854,7,7,0,0,1,1,0,0,0.0,0.0,196.0,196.0,196.0,7.0,7


In [32]:
X_prediction_1 = feature_engineering(X_obs_1)

int64


/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/preprocessor.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_obs["hour"] = X_obs["event_time"].dt.hour
/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/preprocessor.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_obs["dayofweek"] = X_obs["event_time"].dt.dayofweek  # 0=Lundi
/home/sophie/code/AdriMottainai/convertiq/convertiq_py/ml_logic/preprocessor.py:47: SettingWithCopyWarning: 
A value is trying to be set on a c

In [33]:
X_prediction_1.head()

,total_events,total_views,total_carts,total_purchases,n_sessions,n_days_active,has_ever_carted,has_ever_purchased,view_to_cart_ratio,cart_to_purchase_ratio,avg_session_duration,median_session_duration,max_session_duration,avg_events_per_session,max_events_per_session
user_id,,,,,,,,,,,,,,,
551377651,46,38,2,6,7,2,1,1,0.052632,3.0,1553.428571,192.0,9700.0,6.571429,22


In [ ]:
X_prediction_1.to_csv('.csv')

In [ ]:
X_prediction_0.to_csv('.csv')